- Nesse notebook tentarei achar as Regras de Associação que 'trabalham' na NFL. Essas regras funcionam exatamente como a mente humana ao ler aqueles gráficos: procuram cenários de "Se A e B acontecem, então C é quase certo".

- A grande problemática para rodar o algoritmo escolhido, o FP-Growth (o Apriori seria uma alternativa), em dados esportivos é que eles não entendem números contínuos (como 350 jardas ou 24 pontos). Eles entendem eventos categóricos (como uma nota fiscal de mercado). Portanto, precisarei transformar o dataframe de jogos.

- As tags serão baseadas nos 'limites' que analisamos na EDA:
    - Caos: Saldo de Turnovers <= -2
    - Controle: Saldo de Turnovers >= +2
    - Ataque Elite: Pontos Feitos >= 24
    - Defesa Elite: Pontos Sofridos <= 17
    - Mandante: Sim/Não
    - Resultado: Vitória/Derrota

In [6]:
import pandas as pd
import nflreadpy as nfl
from mlxtend.frequent_patterns import fpgrowth, association_rules

# importação dos dados brutos 

ANO_ATUAL = 2026

ultimos_5_anos = list(range(ANO_ATUAL - 5, ANO_ATUAL))

# importando o calendário de jogos (filtrando pré-temporada e pós-temporada)
df_jogos = nfl.load_schedules(ultimos_5_anos).to_pandas()
df_jogos = df_jogos[df_jogos['game_type'] == 'REG'].copy()

# importando as estatísticas dos jogadores
df_estatisticas = nfl.load_player_stats(ultimos_5_anos).to_pandas()

# calculando turnovers e placares

# os turnovers são a soma de interceptações lançadas e fumbles perdidos
df_estatisticas['turnovers_cometidos'] = df_estatisticas['passing_interceptions'] + df_estatisticas['fumbles_lost_total']

# agrupando por jogo e time 
df_turnovers = df_estatisticas.groupby(['game_id', 'team'])['turnovers_cometidos'].sum().reset_index()

# cruzando os dados de jogo com os turnovers
df_to = df_jogos[['game_id', 'home_team', 'away_team', 'home_score', 'away_score']].copy()

# turnovers do Mandante
df_to = df_to.merge(df_turnovers, left_on=['game_id', 'home_team'], right_on=['game_id', 'team'], how='left')
df_to.rename(columns={'turnovers_cometidos': 'home_turnovers'}, inplace=True)
df_to.drop(columns=['team'], inplace=True, errors='ignore')

# turnovers do Visitante
df_to = df_to.merge(df_turnovers, left_on=['game_id', 'away_team'], right_on=['game_id', 'team'], how='left')
df_to.rename(columns={'turnovers_cometidos': 'away_turnovers'}, inplace=True)
df_to.drop(columns=['team'], inplace=True, errors='ignore')

# preenchendo possíveis nulos com 0 (jogos sem turnovers)
df_to.fillna({'home_turnovers': 0, 'away_turnovers': 0}, inplace=True)

# transformação dos dados pro algoritmo entender

# perspectiva do Mandante
df_home = df_to.copy()
df_home['Mandante'] = True
df_home['Vitoria'] = df_home['home_score'] > df_home['away_score']
df_home['Ataque_Elite'] = df_home['home_score'] >= 24
df_home['Defesa_Elite'] = df_home['away_score'] <= 17
df_home['Turnovers_Caos'] = (df_home['home_turnovers'] - df_home['away_turnovers']) >= 2
df_home['Turnovers_Controle'] = (df_home['home_turnovers'] - df_home['away_turnovers']) <= -2

# perspectiva do Visitante
df_away = df_to.copy()
df_away['Mandante'] = False
df_away['Vitoria'] = df_away['away_score'] > df_away['home_score']
df_away['Ataque_Elite'] = df_away['away_score'] >= 24
df_away['Defesa_Elite'] = df_away['home_score'] <= 17
df_away['Turnovers_Caos'] = (df_away['away_turnovers'] - df_away['home_turnovers']) >= 2
df_away['Turnovers_Controle'] = (df_away['away_turnovers'] - df_away['home_turnovers']) <= -2

# empilhando os cenários
df_regras = pd.concat([df_home, df_away])

In [7]:
# algoritmo de data mining --> fp-growth

colunas_foco = ['Mandante', 'Vitoria', 'Ataque_Elite', 'Defesa_Elite', 'Turnovers_Caos', 'Turnovers_Controle']
df_transacoes = df_regras[colunas_foco].copy()

# rodando o fp-growth (a regra precisa aparecer em pelo menos 10% de todos os jogos)
frequencias = fpgrowth(df_transacoes, min_support=0.1, use_colnames=True)

# extraindo e ranqueando as regras (Mínimo de 60% de confiança)
regras = association_rules(frequencias, metric="confidence", min_threshold=0.6)
regras_ordenadas = regras.sort_values(by=['lift', 'confidence'], ascending=[False, False])

# exibindo os padrões minerados
print(f"{'Regra (SE -> ENTÃO)':<65} | Confiança | Lift")
print("-" * 90)
for idx, row in regras_ordenadas.head(15).iterrows():
    antecedentes = ", ".join(list(row['antecedents']))
    consequentes = ", ".join(list(row['consequents']))
    regra_str = f"[{antecedentes}] -> [{consequentes}]"
    print(f"{regra_str:<65} | {row['confidence']:.2f}      | {row['lift']:.2f}")

Regra (SE -> ENTÃO)                                               | Confiança | Lift
------------------------------------------------------------------------------------------
[Defesa_Elite, Ataque_Elite] -> [Vitoria]                         | 1.00      | 2.01
[Defesa_Elite, Turnovers_Controle] -> [Vitoria]                   | 0.96      | 1.92
[Turnovers_Controle, Ataque_Elite] -> [Vitoria]                   | 0.95      | 1.90
[Turnovers_Controle, Vitoria] -> [Defesa_Elite]                   | 0.63      | 1.89
[Turnovers_Controle] -> [Ataque_Elite, Vitoria]                   | 0.65      | 1.88
[Turnovers_Controle] -> [Vitoria]                                 | 0.87      | 1.75
[Defesa_Elite, Mandante] -> [Vitoria]                             | 0.86      | 1.72
[Defesa_Elite] -> [Vitoria]                                       | 0.85      | 1.71
[Turnovers_Controle, Vitoria] -> [Ataque_Elite]                   | 0.75      | 1.66
[Mandante, Ataque_Elite] -> [Vitoria]                      

- A Condição de Perfeição (Confiança 1.00): A regra '[Defesa_Elite, Ataque_Elite] -> [Vitoria]' atingiu a perfeição absoluta. Bater simultaneamente os 24 pontos a favor e segurar o adversário em até 17 dobra a chance base de vitória (Lift 2.01) e torna a derrota estatisticamente nula na sua amostra. O que é óbvio já que 24 é maior que 17, então o modelo apenas descobriu isso.

- O Valor Isolado do Caos: Ter apenas o Turnovers_Controle (saldo favorável) garante a vitória em 87% das vezes. Quando você acopla a proteção da bola a uma unidade de elite (seja Ataque ou Defesa), a confiança salta imediatamente para 95% e 96%. Controlar a bola é o maior catalisador do futebol americano.

- A Pequena Vantagem Defensiva: A regra isolada '[Defesa_Elite] -> [Vitoria]' tem 85% de confiança, superando a regra '[Ataque_Elite] -> [Vitoria]', que fica em 77%. Isso significa que, na ausência de outros fatores combinados, manter o adversário pontuando pouco é uma apólice de seguro ligeiramente mais forte do que ter um ataque explosivo.

- O Mando de Campo como "Buff": A tag Mandante não apareceu como regra isolada com mais de 60% de confiança, o que comprova o que vimos na EDA (a média geral beira os 54%). Além disso, quando associada a uma Defesa ou Ataque de elite, o Fator Casa mostrou que não atua como um multiplicador poderoso, empurrando as confianças de ataque, de 77% para 79%, e de defesa, de 85% para 86%, aumentando apenas 1 ou 2 porcento.

- Temos aqui um clássico problema de vazamento de dados, especificamente na regra de associação com confiança 1. Se for colocado o resultado (Vitória) no mesmo balaio das variáveis diretas que o calculam (Pontos feitos e sofrido), é obvio que fazer mais pontos do que sofre dá a vitória sempre, isso não traz valor/insight. Para extrair insights que não sejam óbvios, será preciso alimentar o algoritmo apenas com as métricas de performance que acontecem durante o jogo, retirando o placar dos antecedentes.

- Adições no lugar do placar:
    - Ataque_Aereo_Forte (Pass Yards > 250).
    - Jogo_Terrestre_Dominante (Rush Yards > 120).
    - Tempo de Posse (Controle de Relógio): medir controle de relógio pelo volume de corridas (carries). Um time que corre 30 ou mais vezes na NFL é um time que está mastigando o cronômetro e mantendo o ataque adversário no banco.
    - Aspecto Defensivo (Pass Rush): A estatística de sacks na base ofensiva mede quantos sacks o Quarterback sofreu. Se o time visitante sofreu 3 ou mais sacks, significa que a defesa do mandante engoliu a linha ofensiva deles.

In [10]:
# agregando as novas variaveis

df_agrupado_novos = df_estatisticas.groupby(['game_id', 'team']).agg(
    passing_yards=('passing_yards', 'sum'),
    rushing_yards=('rushing_yards', 'sum'),
    sacks_sofridos=('sacks_suffered', 'sum'), 
    corridas=('carries', 'sum') 
).reset_index()

# usando uma cópia do df_to que já possui os placares e turnovers calculados
df_regras_base = df_to.copy()

# merge das novas estatísticas para o Mandante
df_regras_base = df_regras_base.merge(df_agrupado_novos, left_on=['game_id', 'home_team'], right_on=['game_id', 'team'], how='left')
df_regras_base.rename(columns={
    'passing_yards': 'home_pass_yds', 
    'rushing_yards': 'home_rush_yds', 
    'sacks_sofridos': 'home_sacks', 
    'corridas': 'home_runs'
}, inplace=True)
df_regras_base.drop(columns=['team'], inplace=True, errors='ignore')

# merge das novas estatísticas para o Visitante
df_regras_base = df_regras_base.merge(df_agrupado_novos, left_on=['game_id', 'away_team'], right_on=['game_id', 'team'], how='left')
df_regras_base.rename(columns={
    'passing_yards': 'away_pass_yds', 
    'rushing_yards': 'away_rush_yds', 
    'sacks_sofridos': 'away_sacks', 
    'corridas': 'away_runs'
}, inplace=True)
df_regras_base.drop(columns=['team'], inplace=True, errors='ignore')

df_regras_base.fillna(0, inplace=True)

# transformação dos dados

# perspectiva do Mandante
df_home = df_regras_base.copy()
df_home['Mandante'] = True
df_home['Vitoria'] = df_home['home_score'] > df_home['away_score']
df_home['Pass_Forte'] = df_home['home_pass_yds'] >= 250
df_home['Rush_Forte'] = df_home['home_rush_yds'] >= 120
df_home['Turnovers_Controle'] = (df_home['home_turnovers'] - df_home['away_turnovers']) <= -2
df_home['Controle_Relogio'] = df_home['home_runs'] >= 30
df_home['Pass_Rush_Elite'] = df_home['away_sacks'] >= 3  # Adversário sofreu 3+ sacks = Defesa dominou

# perspectiva do Visitante
df_away = df_regras_base.copy()
df_away['Mandante'] = False
df_away['Vitoria'] = df_away['away_score'] > df_away['home_score']
df_away['Pass_Forte'] = df_away['away_pass_yds'] >= 250
df_away['Rush_Forte'] = df_away['away_rush_yds'] >= 120
df_away['Turnovers_Controle'] = (df_away['away_turnovers'] - df_away['home_turnovers']) <= -2
df_away['Controle_Relogio'] = df_away['away_runs'] >= 30 
df_away['Pass_Rush_Elite'] = df_away['home_sacks'] >= 3 

# empilhando os cenários
df_regras = pd.concat([df_home, df_away])

In [11]:
# fp-growth

colunas_foco = ['Mandante', 'Vitoria', 'Pass_Forte', 'Rush_Forte', 'Turnovers_Controle', 'Controle_Relogio', 'Pass_Rush_Elite']
df_transacoes = df_regras[colunas_foco].copy()

# rodando o algoritmo
frequencias = fpgrowth(df_transacoes, min_support=0.08, use_colnames=True)
regras = association_rules(frequencias, metric="confidence", min_threshold=0.60)

regras_ordenadas = regras.sort_values(by=['lift', 'confidence'], ascending=[False, False])

# exibição limpa
print(f"{'Regra (SE -> ENTÃO)':<75} | Confiança | Lift")
print("-" * 100)
for idx, row in regras_ordenadas.head(15).iterrows():
    antecedentes = ", ".join(list(row['antecedents']))
    consequentes = ", ".join(list(row['consequents']))
    regra_str = f"[{antecedentes}] -> [{consequentes}]"
    print(f"{regra_str:<75} | {row['confidence']:.2f}      | {row['lift']:.2f}")

Regra (SE -> ENTÃO)                                                         | Confiança | Lift
----------------------------------------------------------------------------------------------------
[Controle_Relogio, Pass_Rush_Elite] -> [Rush_Forte, Vitoria]                | 0.67      | 2.36
[Controle_Relogio, Mandante] -> [Rush_Forte, Vitoria]                       | 0.65      | 2.27
[Rush_Forte, Mandante, Vitoria] -> [Controle_Relogio]                       | 0.80      | 2.23
[Rush_Forte, Vitoria] -> [Controle_Relogio]                                 | 0.78      | 2.19
[Controle_Relogio] -> [Rush_Forte, Vitoria]                                 | 0.62      | 2.19
[Rush_Forte, Turnovers_Controle] -> [Controle_Relogio]                      | 0.78      | 2.19
[Rush_Forte, Vitoria, Pass_Rush_Elite] -> [Controle_Relogio]                | 0.77      | 2.16
[Rush_Forte, Pass_Rush_Elite] -> [Controle_Relogio]                         | 0.69      | 1.94
[Rush_Forte, Mandante] -> [Controle_Relogio]

- O Blueprint Conservador: A combinação '[Controle_Relogio, Turnovers_Controle] -> [Vitoria]' atinge estelares 95% de confiança e um Lift de 1.91. Se um time consegue correr com a bola 30 ou mais vezes e termina com saldo favorável de turnovers, a vitória é quase uma garantia matemática. É a velha cartilha do esporte validada por mineração de dados.

- A Sinergia Máxima (Lift 2.36): A regra no topo absoluto do ranking, '[Controle_Relogio, Pass_Rush_Elite] -> [Rush_Forte, Vitoria]', traduz o conceito de "futebol complementar". Quando a defesa esmaga o QB adversário com 3 ou mais sacks (Pass_Rush_Elite) e o ataque engole o cronômetro (Controle_Relogio), o resultado natural é ultrapassar as 120 jardas terrestres e vencer o jogo. Uma unidade alimenta a outra.

- O Fator Casa Desbloqueado: Nas regras anteriores, foi visto que jogar em casa agregava pouco valor isoladamente. No entanto, a regra '[Controle_Relogio, Mandante] -> [Rush_Forte, Vitoria]' com Lift de 2.27 mostra o gatilho real. Quando o time da casa impõe o jogo terrestre desde o início, a torcida barulhenta força erros do ataque adversário, permitindo que o mandante apenas derreta o cronômetro até o fim.

- A Defesa que Resolve Sozinha: A combinação '[Turnovers_Controle, Pass_Rush_Elite] -> [Vitoria]' (Confiança 0.92) mostra que nem importa muito o que o ataque fez. Se a defesa roubou a bola e sacou o Quarterback 3+ vezes, o time vence em 92% das vezes. A pressão defensiva é suficiente para quebrar o adversário.

- O maior choque dessas regras não são as variáveis que estão nelas, mas a que não apareceu em lugar nenhum: o jogo aéreo. Em uma era em que a NFL é vendida como uma liga dominada por Quarterbacks, que são as superestrelas que ganham mais dinheiro e mais fama/reconhecimento, a variável Pass_Forte (250+ jardas aéreas) desapareceu completamente do Top 15 das associações mais fortes. O algoritmo provou matematicamente que o futebol americano real, aquele que multiplica as chances de vitória, ainda é ditado nas trincheiras.

- Prever vitórias na NFL baseando-se no glamour das jardas aéreas é um erro. O motor da liga é a fisicalidade terrestre atrelada à proteção da bola.